In [40]:
import numpy as np 
import pandas as pd 

X11 = np.load("../data/processed/data_matrix2.npy")
X12 = np.load("../data/processed/data_matrix2_reduced.npy")
X13 = np.load("../data/processed/data_matrix2_reduced1.npy")

y = np.load("../data/processed/y.npy")

import joblib
import json

data_matrix_drop_cols = joblib.load("../artifacts/preprocess/data_matrix2_drop_cols.joblib")

In [41]:
corr_drop_cols = data_matrix_drop_cols['corr_drop_cols']
vif_drop_cols = data_matrix_drop_cols['vif_drop_cols']

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [43]:
xtrain ,  xtest ,ytrain ,ytest = train_test_split(X11 , y , test_size = 0.1 , stratify = y , random_state=42)
xtrain.shape , ytrain.shape , xtest.shape , ytest.shape

((77264, 101), (77264,), (8585, 101), (8585,))

In [44]:
lr = LogisticRegression()
lr.fit(xtrain,ytrain)

print(lr.score(xtrain,ytrain))
print(lr.score(xtest,ytest))

0.6982553323669497
0.6985439720442632


In [45]:
xtrain1 ,  xtest1 ,ytrain1 ,ytest1 = train_test_split(X12 , y , test_size = 0.1 , stratify = y)
xtrain1.shape , ytrain1.shape , xtest1.shape , ytest1.shape

((77264, 92), (77264,), (8585, 92), (8585,))

In [46]:
lr1 = LogisticRegression()
lr1.fit(xtrain1,ytrain1)

print(lr1.score(xtrain1,ytrain1))
print(lr1.score(xtest1,ytest1))

0.7033288465520812
0.7032032615026208


In [47]:
xtrain2 ,  xtest2 ,ytrain2 ,ytest2 = train_test_split(X13 , y , test_size = 0.1 , stratify = y)
xtrain2.shape , ytrain2.shape , xtest2.shape , ytest2.shape

lr2 = LogisticRegression(max_iter = 500)
lr2.fit(xtrain2,ytrain2)

print(lr2.score(xtrain2,ytrain2))
print(lr2.score(xtest2,ytest2))

0.713812383516256
0.7138031450203844


In [48]:
rf1 = RandomForestClassifier(random_state = 42)

rf1.fit(xtrain,ytrain)

print(rf1.score(xtrain,ytrain))
print(rf1.score(xtest,ytest))

0.9999870573617726
0.7129877693651718


In [49]:
rf2 = RandomForestClassifier(random_state=42)
rf2.fit(xtrain1, ytrain1)
print(rf2.score(xtrain1,ytrain1))
print(rf2.score(xtest1,ytest1))

1.0
0.7077460687245195


In [50]:
rf3 = RandomForestClassifier(random_state=42)
rf3.fit(xtrain2, ytrain2)
print(rf3.score(xtrain2,ytrain2))
print(rf3.score(xtest2,ytest2))

0.9999611720853179
0.6986604542807222


In [51]:
xgb1 = XGBClassifier(random_state = 42)
xgb1.fit(xtrain,ytrain-1)

print(xgb1.score(xtrain,ytrain-1))
print(xgb1.score(xtest,ytest-1))

0.7374974114723545
0.7193942923704135


In [52]:
xgb2 = XGBClassifier(random_state = 42)
xgb2.fit(xtrain1,ytrain1-1)

print(xgb2.score(xtrain1,ytrain1-1))
print(xgb2.score(xtest1,ytest1-1))

0.7381704286601781
0.7152009318578917


In [53]:
xgb3 = XGBClassifier(random_state = 42)
xgb3.fit(xtrain2,ytrain2-1)

print(xgb3.score(xtrain2,ytrain2-1))
print(xgb3.score(xtest2,ytest2-1))

0.7371091323255332
0.7139196272568433


In [54]:
joblib.dump( xgb3,"../artifacts/models/xgb.joblib")

['../artifacts/models/xgb.joblib']

ANN model 

In [55]:
import warnings 
warnings.filterwarnings('ignore')

In [56]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout , Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Example input shape: number of features in your dataset
input_dim = (xtrain1.shape[1],)  

# Build the model
model = Sequential([
    Input(shape = input_dim) ,
    Dense(128, activation='relu'),
    Dropout(0.2),                  # helps prevent overfitting
    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(np.unique(y).size, activation='softmax')  # multi-class classification
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # use 'categorical_crossentropy' if one-hot encoding
    metrics=['accuracy']
)

# # Early stopping to prevent overfitting
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True
# )

In [57]:
# Train the model
history = model.fit(
    xtrain1, ytrain1-1,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    # callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.6647 - loss: 1.5225 - val_accuracy: 0.6943 - val_loss: 0.9970
Epoch 2/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6935 - loss: 0.9723 - val_accuracy: 0.6943 - val_loss: 0.9535
Epoch 3/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6937 - loss: 0.9600 - val_accuracy: 0.6943 - val_loss: 0.9549
Epoch 4/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6939 - loss: 0.9548 - val_accuracy: 0.6943 - val_loss: 0.9529
Epoch 5/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6940 - loss: 0.9533 - val_accuracy: 0.6943 - val_loss: 0.9525
Epoch 6/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6940 - loss: 0.9549 - val_accuracy: 0.6943 - val_loss: 0.9521
Epoch 7/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6941 - loss: 0.9519 - val_accuracy: 0.6952 - val_loss: 0.9493
Epoch 8/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6951 - loss: 0

In [58]:
# Evaluate
train_acc = model.evaluate(xtrain1, ytrain1-1, verbose=0)[1]
test_acc = model.evaluate(xtest1, ytest1-1, verbose=0)[1]

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Train Accuracy: 0.7176
Test Accuracy: 0.7100


After Rigorous model creation using various algorithms and using all variants of the processed data for model training, we finally chose the best performing data variant ( data_matrix2_reduced1) . This data variant differs from the other processed data variants (data_matrix and data_matrix1) in only one aspect which is encoding of the supervisor feature. Here we have done simple target encoding where mean csat score of each supervisor is replacing the supervisor . Other than that all three variants are same in all aspects .

However ,on analysis of training using all models and all data variants , it was found that all the data variants were performing nearly equally good for all models . But we have to chose one of the variants , so we chose this one (data_matrix2_reduced1) because it had lower dimensionality compared to the other one where supervisor feature was one hot encoded creating many new dimensions for the input data. 

On model performance , all the models , Logistic regression , Random Forest , XGB and DNN with 4 layers and even DNN with 8 layers were giving nearly similar results on the aspect of model accuracy. The accuracy for all these models was around 70 percentage . All attempts to improve were not enough. Maybe it was due to the lack of informative features in the dataset 


Observation:

“Across all models tested—including Logistic Regression, Random Forest, XGBoost, and multiple ANN architectures—the maximum achievable accuracy plateaued around ~70%. This pattern persisted despite careful preprocessing, hyperparameter tuning, and attempts with deeper network architectures.

Interpretation:

The consistent performance ceiling across diverse model families strongly suggests that the limitation is intrinsic to the dataset rather than the choice of model or feature processing. Possible contributing factors include:

Limited predictive information in the features provided

Class overlap or noise in the labels

Imbalanced or insufficiently informative features

Conclusion:

Therefore, further improvement in model performance is unlikely without additional high-quality data, feature engineering, or domain-specific features. The plateau is a data-driven limitation, not a modeling issue.

In [59]:
model.save("../artifacts/models/ANN_4_layers.h5")

Since our models are not performing well enough for the multiclass classification problem , we will convert the problem into a binary problem where csat scores will be transformed to 0 and 1 . 0 is for csat scores of 1 , 2 ,3 and 1 is for the csat scores of 4 ,5 . 

In [60]:
# X  = X13 
# y = y 

new_y = np.where(y <= 3 , 0 ,1)

In [61]:
np.unique(new_y , return_counts = True)

(array([0, 1]), array([15052, 70797]))

In [62]:
x_train , x_test , y_train , y_test = train_test_split(X12 , new_y , random_state=42 , test_size = 0.1 , stratify = y)

In [63]:
x_train.shape , x_test.shape

((77264, 92), (8585, 92))

In [64]:
# XBG Classifier 

scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
xgb4 = XGBClassifier(scale_pos_weight=scale_pos_weight)

xgb4.fit(x_train , y_train)

print(xgb4.score(x_train , y_train))
print(xgb4.score(x_test , y_test))

0.7679384965831435
0.7480489225393128


In [65]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout , Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Example input shape: number of features in your dataset
input_dim = (x_train.shape[1],)  

# Build the model
model1 = Sequential([
    Input(shape = input_dim) ,
    Dense(128, activation='relu'),
    Dropout(0.3),                  # helps prevent overfitting
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # multi-class classification
])

# Compile the model
model1.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',  # use 'categorical_crossentropy' if one-hot encoding
    metrics=['accuracy']
)

# Early stopping to prevent overfitting
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True
# )

In [67]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

class_weights

{0: np.float64(2.8517014837233337), 1: np.float64(0.6063060093852504)}

In [68]:
# Train the model
history = model1.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    # callbacks=[early_stop],
    class_weight = class_weights,
    verbose=1
)

Epoch 1/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.5220 - loss: 1.0200 - val_accuracy: 0.5888 - val_loss: 0.6969
Epoch 2/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.5607 - loss: 0.7048 - val_accuracy: 0.7994 - val_loss: 0.6899
Epoch 3/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.5742 - loss: 0.6954 - val_accuracy: 0.8288 - val_loss: 0.6918
Epoch 4/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.4591 - loss: 0.6958 - val_accuracy: 0.1712 - val_loss: 0.6973
Epoch 5/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.3121 - loss: 0.6944 - val_accuracy: 0.8288 - val_loss: 0.6929
Epoch 6/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.3643 - loss: 0.6940 - val_accuracy: 0.8288 - val_loss: 0.6900
Epoch 7/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.4774 - loss: 0.6980 - val_accuracy: 0.1712 - val_loss: 0.6977
Epoch 8/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.3833 - loss: 0

In [77]:
# Evaluate
train_acc = model1.evaluate(x_train, y_train, verbose=0)[1]
test_acc = model1.evaluate(x_test, y_test, verbose=0)[1]

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Train Accuracy: 0.6466
Test Accuracy: 0.6560


In [69]:
model1.save("../artifacts/models/ANN_4_layers_2.h5")

In [72]:
x_train1 , x_test1 , y_train1 , y_test1 = train_test_split(X12 , new_y , random_state=42 , test_size = 0.1 , stratify = y)

In [73]:
x_train1.shape , x_test1.shape

((77264, 92), (8585, 92))

6 layered ann model

In [75]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout , Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Example input shape: number of features in your dataset
input_dim = (x_train1.shape[1],)  

# Build the model
model2 = Sequential([
    Input(shape = input_dim) ,
    Dense(256, activation='relu'),
    Dropout(0.3),                  # helps prevent overfitting
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  
])

# Compile the model
model2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',  # use 'categorical_crossentropy' if one-hot encoding
    metrics=['accuracy']
)

# # Early stopping to prevent overfitting
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True
# )

In [74]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train1),
    y=y_train1
)
class_weights = dict(enumerate(class_weights))

class_weights

{0: np.float64(2.8517014837233337), 1: np.float64(0.6063060093852504)}

In [76]:
# Train the model
history = model2.fit(
    x_train1, y_train1,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    # callbacks=[early_stop],
    class_weight = class_weights,
    verbose=1
)

Epoch 1/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.5344 - loss: 0.8584 - val_accuracy: 0.7971 - val_loss: 0.6832
Epoch 2/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4924 - loss: 0.6985 - val_accuracy: 0.8288 - val_loss: 0.6848
Epoch 3/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6207 - loss: 0.6953 - val_accuracy: 0.1712 - val_loss: 0.7053
Epoch 4/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.3628 - loss: 0.6941 - val_accuracy: 0.1712 - val_loss: 0.6980
Epoch 5/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.3508 - loss: 0.6945 - val_accuracy: 0.1746 - val_loss: 0.6974
Epoch 6/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.4351 - loss: 0.6946 - val_accuracy: 0.1712 - val_loss: 0.7092
Epoch 7/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.4979 - loss: 0.6941 - val_accuracy: 0.1712 - val_loss: 0.7051
Epoch 8/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.4585 - loss: 0

In [88]:
# Evaluate
train_acc = model2.evaluate(x_train1, y_train1, verbose=0)[1]
test_acc = model2.evaluate(x_test1, y_test1, verbose=0)[1]

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Train Accuracy: 0.8247
Test Accuracy: 0.8247


model2 is the best performer so far 

In [79]:
model2.save("../artifacts/models/ANN_6_layers.h5")

8 layered ANN Model

In [80]:
from tensorflow.keras.metrics import Precision, Recall


# Example input shape: number of features in your dataset
input_dim = (x_train1.shape[1],)  

# Build the 8-layer ANN model
model3 = Sequential([
    Input(shape=input_dim),
    Dense(512, activation='relu'),   # 1st hidden layer
    Dropout(0.3),
    Dense(256, activation='relu'),   # 2nd hidden layer
    Dropout(0.3),
    Dense(128, activation='relu'),   # 3rd hidden layer
    Dropout(0.2),
    Dense(64, activation='relu'),    # 4th hidden layer
    Dropout(0.2),
    Dense(32, activation='relu'),    # 5th hidden layer
    Dropout(0.1),
    Dense(16, activation='relu'),    # 6th hidden layer
    Dense(8, activation='relu'),     # 7th hidden layer
    Dense(1, activation='sigmoid')   # Output layer
])

# Compile the model
model3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

# Optional: Early stopping
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True
# )


In [81]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train1),
    y=y_train1
)
class_weights = dict(enumerate(class_weights))

class_weights

{0: np.float64(2.8517014837233337), 1: np.float64(0.6063060093852504)}

In [82]:
# Train the model
history = model3.fit(
    x_train1, y_train1,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    # callbacks=[early_stop],
    class_weight = class_weights,
    verbose=1
)

Epoch 1/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.2853 - loss: 0.7157 - precision: 0.8191 - recall: 0.1705 - val_accuracy: 0.1712 - val_loss: 0.6958 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.4373 - loss: 0.6940 - precision: 0.8228 - recall: 0.4044 - val_accuracy: 0.1712 - val_loss: 0.6968 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.2254 - loss: 0.6939 - precision: 0.8261 - recall: 0.0763 - val_accuracy: 0.8288 - val_loss: 0.6891 - val_precision: 0.8288 - val_recall: 1.0000
Epoch 4/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.5586 - loss: 0.6947 - precision: 0.8237 - recall: 0.5909 - val_accuracy: 0.1712 - val_loss: 0.7010 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/100
2174/2174 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.3800 - loss: 0.6939 - precision: 0.8215 - recall: 0.316

In [89]:
# Evaluate
train_acc = model3.evaluate(x_train1, y_train1, verbose=0)[1]
test_acc = model3.evaluate(x_test1, y_test1, verbose=0)[1]

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Train Accuracy: 0.1753
Test Accuracy: 0.1753


In [90]:
model3.save("../artifacts/models/ANN_8_layers.h5")